In [1]:
import pandas as pd
import numpy as np
import os  #file and folder paths

In [2]:
YEAR = 2025
FILE_PATH = "../data/raw/2025.csv" 

In [58]:
df_clean_test=pd.read_csv(
    FILE_PATH,
    nrows=100000
)
df_clean_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   State           100000 non-null  str    
 1   District        100000 non-null  str    
 2   Market          100000 non-null  str    
 3   Commodity       100000 non-null  str    
 4   Variety         100000 non-null  str    
 5   Grade           100000 non-null  str    
 6   Arrival_Date    100000 non-null  str    
 7   Min_Price       100000 non-null  float64
 8   Max_Price       100000 non-null  float64
 9   Modal_Price     100000 non-null  float64
 10  Commodity_Code  100000 non-null  int64  
dtypes: float64(3), int64(1), str(7)
memory usage: 8.4 MB


Function to standardize the column names 

In [60]:
def standardize_columns(df):
    df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))
    return df
df_clean_test=standardize_columns(df_clean_test)
df_clean_test.columns

Index(['state', 'district', 'market', 'commodity', 'variety', 'grade',
       'arrival_date', 'min_price', 'max_price', 'modal_price',
       'commodity_code'],
      dtype='str')

Cleaning the column names

In [ ]:
text_columns = ["state","district","market","commodity","variety","grade"] #string columns 
price_columns = ["min_price","max_price","modal_price"]  #Integer columns prizes 
df_clean_test[price_columns].describe()


Here 25,50 and 75% represents the quartiles means cut-off point in sorted data. 
most frequently occurring price
As normally prize should be min<modal<max

Cleaning the text(String) columns the each attribute means text in it 

In [61]:

def clean_text_columns(df, text_columns):
    for column in text_columns:
        df[column]=df[column].astype("string").str.strip()
        #.astype("string")Ensures that the column is treated as text.
        # .str.strip()Removes unnecessary spaces from the beginning and end
        return df
df_clean_test = clean_text_columns(df_clean_test,text_columns)

Converting the date from string to date format 

In [62]:
def clean_date_column(df,date_column):
    df[date_column]=pd.to_datetime(df[date_column],format="%Y-%m-%d",
                                                errors="coerce")#if error occurs in the value it replaces it with the NAN 
    return df
df_clean_test=clean_date_column(df_clean_test,"arrival_date")

Creating extra columns for future analysis

In [63]:
def clean_date_features(df,date_column):
    df["year"]=df[date_column].dt.year
    df["month"]=df[date_column].dt.month
    df["month_name"] = (df[date_column].dt.month_name())
    df["month"].head
    return df
df_clean_test=clean_date_features(df_clean_test,"arrival_date")
df_clean_test["month"].head()


0    1
1    1
2    1
3    1
4    1
Name: month, dtype: int32

In [64]:
def check_order_price(df):
    invalid_price_order = (
        (df["min_price"] > df["modal_price"]) |
        (df["modal_price"] > df["max_price"])
    )
    return invalid_price_order

Checking negative and zero prices

In [65]:
def check_negative_price(df,price_columns):
    negative_price_count = (df[price_columns] < 0).sum()
    return negative_price_count

In [66]:
def check_zero_prices(df,price_columns):
    zero_price_count = (df[price_columns] == 0).sum()
    return zero_price_count

In [ ]:
zero_price_count = check_zero_prices(df_clean_test,price_columns)
zero_price_count

min_price      27
max_price       0
modal_price     0
dtype: int64

In [37]:
zero_price_mask = ((df_clean_test["min_price"] == 0) |(df_clean_test["max_price"] == 0) |(df_clean_test["modal_price"] == 0))
df_clean_test[zero_price_mask].head(5)

,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price,commodity_code,year,month,month_name
302,Tamil Nadu,Theni,Theni (Uzhavar Sandhai ),Chow Chow,Chow Chow,Local,2025-01-01,0.0,2400.0,2400.0,167,2025,1,January
303,Tamil Nadu,Theni,Theni (Uzhavar Sandhai ),Cluster beans,Cluster Beans,Local,2025-01-01,0.0,4000.0,4000.0,80,2025,1,January
304,Tamil Nadu,Theni,Theni (Uzhavar Sandhai ),Coconut,Coconut,Local,2025-01-01,0.0,6000.0,6000.0,138,2025,1,January
305,Tamil Nadu,Theni,Theni (Uzhavar Sandhai ),Colacasia,Colacasia,Local,2025-01-01,0.0,4500.0,4500.0,318,2025,1,January
306,Tamil Nadu,Theni,Theni (Uzhavar Sandhai ),Coriander (Leaves),I Sort,Local,2025-01-01,0.0,3200.0,3200.0,43,2025,1,January


As here we can see mostly the min price has the zero values so insted of daleting it directly we are going ot make the column "has zro value" then according to it we can decide is it genuiely the correct data or false value 


In [67]:
df_clean_test["Has_Zero_Price"] = (df_clean_test[price_columns] == 0).any(axis=1)

In [68]:
for column in text_columns:
    print(f"\n{column}")
    print("Unique values:", df_clean_test[column].nunique())
    print(df_clean_test[column].value_counts().head(10))


state
Unique values: 27
state
Tamil Nadu        38508
Uttar Pradesh     15109
Kerala             8289
Maharashtra        6078
Madhya Pradesh     5352
Gujarat            3580
West Bengal        3040
Punjab             2988
Karnataka          2881
Haryana            2772
Name: count, dtype: Int64

district
Unique values: 509
district
Salem              3256
Coimbatore         2363
Thiruvannamalai    1903
Madurai            1871
Vellore            1719
Theni              1686
Namakkal           1518
Virudhunagar       1429
Chengalpattu       1394
Krishnagiri        1336
Name: count, dtype: int64

market
Unique values: 2259
market
Tiruvannamalai (Uzhavar Sandhai )     410
Hosur (Uzhavar Sandhai )              402
Anna nagar (Uzhavar Sandhai )         388
RSPuram (Uzhavar Sandhai )            388
Vellore                               386
Kahithapattarai (Uzhavar Sandhai )    368
Thathakapatti (Uzhavar Sandhai )      361
Sooramangalam (Uzhavar Sandhai )      356
Chokkikulam (Uzhavar Sandhai

To tell where the zero ,negative ,or inorder prize is present 

In [69]:
def add_price_quality_flags(df, price_columns):
    
    df["invalid_price_order"] = (
        (df["min_price"] > df["modal_price"]) |
        (df["modal_price"] > df["max_price"])
    )
    
    df["has_negative_price"] = (
        df[price_columns] < 0
    ).any(axis=1)
    
    df["has_zero_price"] = (
        df[price_columns] == 0
    ).any(axis=1)
    
    return df

### Writinh the clean chunkwhich will comines the cleaning functions togetherly and execute togetherly 

In [70]:
def clean_chunk(df):
    df = standardize_columns(df)
    df = clean_text_columns(df,text_columns)
    df = clean_date_column(df,"arrival_date")
    df = clean_date_features(df,"arrival_date")
    df=add_price_quality_flags(df, price_columns)
    return df


### For validating the data after cleaning 

In [71]:
def validate_chunk(df):

    quality_report = {
        "rows": len(df),
        "missing_dates": df["arrival_date"].isna().sum(),
        "invalid_price_order": df["invalid_price_order"].sum(),
        "negative_price_rows": df["has_negative_price"].sum(),
        "zero_price_rows": df["has_zero_price"].sum()
    }

    return quality_report

In [72]:
df_clean_test = clean_chunk(df_clean_test)


In [73]:
quality_report = validate_chunk(
    df_clean_test
)

quality_report

{'rows': 100000,
 'missing_dates': np.int64(0),
 'invalid_price_order': np.int64(38),
 'negative_price_rows': np.int64(0),
 'zero_price_rows': np.int64(144)}